# 1교시: 환경 설정과 Docker 네트워크

## 학습 목표
- Docker Compose로 PostgreSQL + pgAdmin 환경 구축
- **호스트 ↔ 컨테이너** vs **컨테이너 ↔ 컨테이너** 네트워크 차이 이해
- VSCode SQLTools로 DB 연결

---

## 1. 데이터 엔지니어와 SQL

### 데이터 엔지니어가 하는 일
```
📥 데이터 수집 → 🔄 데이터 처리 → 📊 데이터 저장 → 📈 데이터 제공
```

- **SQL**: 데이터 엔지니어의 기본 언어
- 모든 데이터 파이프라인에서 SQL 사용
  - Kafka → DB 저장 (INSERT)
  - Spark SQL (대용량 처리)
  - Airflow DAG (스케줄링된 쿼리)

### 이번 2일의 목표
| Day | 내용 |
|-----|------|
| Day 1 | SELECT, WHERE, GROUP BY 등 기본 조회 |
| Day 2 | JOIN, 서브쿼리, 윈도우 함수 |

---
## 2. 실습 환경 구성

### 프로젝트 폴더 생성
터미널에서 실행:
```bash
mkdir -p ~/sql-practice/init
cd ~/sql-practice
```

### docker-compose.yml 작성

`~/sql-practice/docker-compose.yml` 파일 생성:

```yaml
services: # 실행할 서비스(컨테이너)들의 묶음입니다.

  # 1. 포스트그레스 데이터베이스 서비스 설정
  postgres:
    image: postgres:15 # 사용할 이미지의 이름과 버전(15버전)입니다.
    container_name: delivery_db # 실행될 컨테이너의 별명을 'delivery_db'로 지정합니다.

    environment: # 컨테이너 내부에서 사용할 환경 변수(설정값)들입니다.
      POSTGRES_USER: student      # 데이터베이스 접속용 사용자 아이디
      POSTGRES_PASSWORD: student123 # 사용자 비밀번호
      POSTGRES_DB: delivery       # 처음에 자동으로 생성할 데이터베이스 이름

    ports:
      - "5432:5432" # [내 컴퓨터 포트]:[컨테이너 내부 포트]를 연결합니다.
                    # 외부 프로그램(DBeaver 등)에서 접속할 때 5432 포트를 사용합니다.

    networks:
      - db_network # 아래에서 정의한 'db_network'라는 가상 네트워크에 소속시킵니다.

    volumes:
      # [내 컴퓨터 경로]:[컨테이너 내부 경로]
      # 내 컴퓨터의 ./init 폴더에 SQL 파일을 넣어두면,
      # 컨테이너가 처음 실행될 때 해당 SQL들을 읽어서 테이블을 자동으로 만듭니다.
      - ./init:/docker-entrypoint-initdb.d

  # 2. pgAdmin (웹 기반 데이터베이스 관리 도구) 서비스 설정
  pgadmin:
    image: dpage/pgadmin4 # pgAdmin 공식 이미지를 사용합니다.
    container_name: pgadmin

    environment:
      PGADMIN_DEFAULT_EMAIL: admin@example.com # 웹 관리 페이지 로그인용 이메일
      PGADMIN_DEFAULT_PASSWORD: admin          # 웹 관리 페이지 로그인용 비밀번호

    ports:
      - "8080:80" # 웹 브라우저에서 'localhost:8080'으로 접속할 수 있게 설정합니다.

    networks:
      - db_network # 데이터베이스와 같은 네트워크에 있어야 서로 통신이 가능합니다.

# 가상 네트워크 설정
networks:
  db_network: # 서비스들이 서로를 알아볼 수 있도록 통로를 만들어줍니다.
    driver: bridge # 가장 기본적인 네트워크 방식인 'bridge' 모드를 사용합니다.
```

> 📖 **참고**: `docker-entrypoint-initdb.d`에 넣은 SQL 파일은 컨테이너 시작 시 자동 실행

### 샘플 데이터 생성

`~/sql-practice/init/01_schema.sql` 파일 생성:

```sql
-- 고객 테이블
CREATE TABLE users (
    user_id SERIAL PRIMARY KEY,
    name VARCHAR(50) NOT NULL,
    region VARCHAR(20) NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- 식당 테이블
CREATE TABLE restaurants (
    restaurant_id SERIAL PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    category VARCHAR(20) NOT NULL,
    region VARCHAR(20) NOT NULL
);

-- 주문 테이블
CREATE TABLE orders (
    order_id SERIAL PRIMARY KEY,
    user_id INTEGER REFERENCES users(user_id),
    restaurant_id INTEGER REFERENCES restaurants(restaurant_id),
    total_amount INTEGER NOT NULL,
    status VARCHAR(20) DEFAULT 'pending',
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- 주문 상세 테이블
CREATE TABLE order_items (
    item_id SERIAL PRIMARY KEY,
    order_id INTEGER REFERENCES orders(order_id),
    menu_name VARCHAR(100) NOT NULL,
    quantity INTEGER NOT NULL,
    price INTEGER NOT NULL
);
```

`~/sql-practice/init/02_seed_data.sql` 파일 생성:

```sql
-- 지역 목록
-- '강남', '서초', '송파', '마포', '영등포', '강서', '성북', '노원', '중구', '종로'

-- 고객 100명 생성
INSERT INTO users (name, region) VALUES
('김민수', '강남'), ('이영희', '서초'), ('박준호', '송파'), ('최지연', '마포'),
('정현우', '영등포'), ('강수진', '강서'), ('윤도현', '성북'), ('임서연', '노원'),
('한지민', '중구'), ('오승현', '종로'), ('신예린', '강남'), ('권태우', '서초'),
('조미래', '송파'), ('배성훈', '마포'), ('류지원', '영등포'), ('홍길동', '강서'),
('문채원', '성북'), ('장우진', '노원'), ('이하늘', '중구'), ('김도윤', '종로'),
('박소연', '강남'), ('최현석', '서초'), ('정다은', '송파'), ('강민호', '마포'),
('윤서아', '영등포'), ('임재현', '강서'), ('한승우', '성북'), ('오지현', '노원'),
('신동욱', '중구'), ('권나연', '종로'), ('조성민', '강남'), ('배유나', '서초'),
('류현진', '송파'), ('홍수빈', '마포'), ('문재원', '영등포'), ('장서현', '강서'),
('이준혁', '성북'), ('김서윤', '노원'), ('박민재', '중구'), ('최예진', '종로'),
('정우성', '강남'), ('강다현', '서초'), ('윤지호', '송파'), ('임수아', '마포'),
('한도윤', '영등포'), ('오현우', '강서'), ('신민서', '성북'), ('권재훈', '노원'),
('조예린', '중구'), ('배지훈', '종로'), ('류서연', '강남'), ('홍민준', '서초'),
('문지영', '송파'), ('장현우', '마포'), ('이소율', '영등포'), ('김태현', '강서'),
('박지민', '성북'), ('최승현', '노원'), ('정서윤', '중구'), ('강재원', '종로'),
('윤예진', '강남'), ('임도현', '서초'), ('한서아', '송파'), ('오태윤', '마포'),
('신지현', '영등포'), ('권승우', '강서'), ('조민지', '성북'), ('배현우', '노원'),
('류다은', '중구'), ('홍서현', '종로'), ('문승민', '강남'), ('장예린', '서초'),
('이재원', '송파'), ('김유진', '마포'), ('박도현', '영등포'), ('최서아', '강서'),
('정민준', '성북'), ('강지은', '노원'), ('윤태우', '중구'), ('임예나', '종로'),
('한민서', '강남'), ('오지훈', '서초'), ('신서윤', '송파'), ('권도윤', '마포'),
('조현석', '영등포'), ('배미래', '강서'), ('류재훈', '성북'), ('홍예진', '노원'),
('문도현', '중구'), ('장민지', '종로'), ('이승아', '강남'), ('김현우', '서초'),
('박예린', '송파'), ('최재현', '마포'), ('정지수', '영등포'), ('강서윤', '강서'),
('윤민재', '성북'), ('임지현', '노원'), ('한태훈', '중구'), ('오예나', '종로');

-- 식당 20개 생성
INSERT INTO restaurants (name, category, region) VALUES
('황금치킨', '치킨', '강남'), ('피자나라', '피자', '서초'), ('중화반점', '중식', '송파'),
('한식당', '한식', '마포'), ('분식천국', '분식', '영등포'), ('치킨매니아', '치킨', '강서'),
('도미노피자', '피자', '성북'), ('짬뽕하우스', '중식', '노원'), ('시골밥상', '한식', '중구'),
('떡볶이왕', '분식', '종로'), ('굽네치킨', '치킨', '강남'), ('피자헛', '피자', '서초'),
('홍콩반점', '중식', '송파'), ('된장마을', '한식', '마포'), ('신전떡볶이', '분식', '영등포'),
('BBQ치킨', '치킨', '강서'), ('파파존스', '피자', '성북'), ('양자강', '중식', '노원'),
('전주비빔밥', '한식', '중구'), ('죠스떡볶이', '분식', '종로');

-- 주문 500건 생성 (2025년 11월 ~ 12월)
INSERT INTO orders (user_id, restaurant_id, total_amount, status, created_at)
SELECT
    (random() * 99 + 1)::int,
    (random() * 19 + 1)::int,
    (random() * 45000 + 5000)::int,
    (ARRAY['completed', 'completed', 'completed', 'completed', 'pending', 'cancelled'])[floor(random() * 6 + 1)::int],
    timestamp '2025-11-01' + (random() * 60) * interval '1 day' + (random() * 86400) * interval '1 second'
FROM generate_series(1, 500);

-- 주문 상세 (주문당 1~3개 메뉴)
INSERT INTO order_items (order_id, menu_name, quantity, price)
SELECT
    o.order_id,
    (ARRAY['후라이드치킨', '양념치킨', '마르게리타', '페퍼로니', '짜장면', '짬뽕',
           '비빔밥', '된장찌개', '떡볶이', '순대', '튀김', '김밥'])[floor(random() * 12 + 1)::int],
    (random() * 2 + 1)::int,
    (random() * 15000 + 5000)::int
FROM orders o
CROSS JOIN generate_series(1, (random() * 2 + 1)::int);
```

---
## 3. Docker 컨테이너 실행

터미널에서:
```bash
cd ~/sql-practice
docker compose up -d
```

### 상태 확인
```bash
# 실행 중인 컨테이너 확인
docker ps

# 예상 출력:
# CONTAINER ID   IMAGE              PORTS                    NAMES
# abc123...      postgres:15        0.0.0.0:5432->5432/tcp   delivery_db
# def456...      dpage/pgadmin4     0.0.0.0:8080->80/tcp     pgadmin
```

### 네트워크 확인
```bash
docker network ls

# sql-practice_db_network 네트워크가 생성되어 있어야 함
```

---
## 4. 🚨 의도적 오류: Docker 네트워크 이해하기

### pgAdmin에서 연결 시도 (실패 케이스)

1. 브라우저에서 http://localhost:8080 접속
2. 로그인: `admin@example.com` / `admin`
3. Add New Server 클릭
4. General 탭: Name = `delivery`
5. Connection 탭:
   - Host: **`localhost`** ← ❌ 이게 문제!
   - Port: 5432
   - Database: delivery
   - Username: student
   - Password: student123

### 결과: 연결 실패! 🔴
```
connection to server at "localhost" (127.0.0.1), port 5432 failed:
Connection refused
```

### 왜 실패했을까?
```
┌─────────────────────────────────────────────────────────────┐
│                    Docker 네트워크 구조                      │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│   호스트 (내 맥/윈도우)                                      │
│   └── localhost:5432 → postgres 컨테이너                    │
│   └── localhost:8080 → pgadmin 컨테이너                     │
│                                                             │
│   ─────────────────────────────────────────────────────     │
│                                                             │
│   Docker 네트워크 (db_network)                              │
│   ├── postgres 컨테이너                                     │
│   │   └── 내부에서 localhost = 자기 자신                    │
│   │                                                         │
│   └── pgadmin 컨테이너                                      │
│       └── 내부에서 localhost = 자기 자신 (postgres 아님!)   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

- pgAdmin 컨테이너 안에서 `localhost` = pgAdmin 자신
- postgres 컨테이너에 연결하려면 **서비스명** 사용!

### 올바른 연결 방법 (성공 케이스) ✅

Connection 탭 수정:
- Host: **`postgres`** ← 서비스명 사용!
- Port: 5432
- Database: delivery
- Username: student
- Password: student123

### 연결 성공! 🟢

> **핵심**: 컨테이너 ↔ 컨테이너 통신은 **서비스명**으로!

---
## 5. VSCode SQLTools 연결

### SQLTools 설치
1. VSCode 확장 탭 (Ctrl+Shift+X)
2. "SQLTools" 검색 → 설치
3. "SQLTools PostgreSQL" 드라이버도 설치

### 새 연결 추가
1. 왼쪽 SQLTools 아이콘 클릭
2. Add New Connection
3. PostgreSQL 선택
4. 설정:
   - Connection Name: delivery
   - Server Address: **`localhost`** ← 호스트에서 접속하므로 localhost OK!
   - Port: 5432
   - Database: delivery
   - Username: student
   - Password: student123

### 왜 여기선 localhost가 되나요?
```
VSCode (호스트에서 실행)
    │
    └── localhost:5432 ──→ Docker가 포트 포워딩 ──→ postgres 컨테이너
```
- VSCode는 **호스트**에서 실행됨
- Docker가 호스트의 5432 포트를 컨테이너로 연결해줌

---
## 6. 핵심 정리

| 상황 | 호스트 사용 |
|------|------------|
| 호스트 → 컨테이너 | `localhost:포트` |
| 컨테이너 → 컨테이너 | `서비스명:포트` |

### 기억할 것
1. `docker compose up -d`로 컨테이너 실행
2. `docker ps`로 상태 확인
3. 컨테이너끼리는 **서비스명**으로 통신
4. 호스트에서는 **localhost + 포트**로 접속

---

## 다음 교시 예고
SELECT, WHERE 문으로 데이터 조회 시작!

---


# 2교시: SELECT와 WHERE - 데이터 조회 기초

## 학습 목표
- 테이블 구조 파악하기
- SELECT로 원하는 데이터 조회
- WHERE로 조건 필터링

---

## 1. 테이블 구조 확인

### 우리가 사용할 데이터: 배달 서비스 "배달왕"

```
users (고객 100명)
├── user_id    - 고객 번호 (PK)
├── name       - 이름
├── region     - 지역
└── created_at - 가입일

restaurants (식당 20개)
├── restaurant_id - 식당 번호 (PK)
├── name          - 상호명
├── category      - 카테고리 (치킨, 피자, 중식, 한식, 분식)
└── region        - 지역

orders (주문 500건)
├── order_id      - 주문 번호 (PK)
├── user_id       - 주문한 고객 (FK)
├── restaurant_id - 주문한 식당 (FK)
├── total_amount  - 총 금액
├── status        - 상태 (completed, pending, cancelled)
└── created_at    - 주문 시간
```

### 테이블 목록 보기

VSCode SQLTools에서 실행:

```sql
-- 테이블 목록 확인
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public';
```

결과:
```
table_name
------------
users
restaurants
orders
order_items
```

### 테이블 구조 보기

```sql
-- users 테이블 컬럼 정보
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'users';
```

결과:
```
column_name | data_type
------------|----------------------------
user_id     | integer
name        | character varying
region      | character varying
created_at  | timestamp without time zone
```

---
## 2. SELECT 기초

### 전체 데이터 조회

```sql
SELECT * FROM users;
```

결과 (100건):
```
user_id | name   | region | created_at
--------|--------|--------|--------------------
1       | 김민수 | 강남   | 2025-01-07 10:00:00
2       | 이영희 | 서초   | 2025-01-07 10:00:00
...
```

### 🚨 의도적 오류: LIMIT 없이 대용량 테이블 조회

```sql
-- orders 테이블 전체 조회 (500건)
SELECT * FROM orders;
```

결과: 500건이 한꺼번에 출력됨!

> ⚠️ **실무 주의**: 운영 DB에서 LIMIT 없이 조회하면 수백만 건이 쏟아질 수 있음

### 올바른 방법: LIMIT 사용

```sql
SELECT * FROM orders LIMIT 10;
```

결과 (10건만):
```
order_id | user_id | restaurant_id | total_amount | status    | created_at
---------|---------|---------------|--------------|-----------|--------------------
1        | 42      | 7             | 23500        | completed | 2025-11-15 14:30:00
2        | 15      | 3             | 18000        | completed | 2025-11-02 19:45:00
...
```

### 특정 컬럼만 조회

```sql
-- 필요한 컬럼만 선택
SELECT user_id, total_amount, status
FROM orders
LIMIT 10;
```

결과:
```
user_id | total_amount | status
--------|--------------|----------
42      | 23500        | completed
15      | 18000        | completed
...
```

---
## 3. WHERE - 조건 필터링

### 기본 조건

```sql
-- 완료된 주문만 조회
SELECT * FROM orders
WHERE status = 'completed'
LIMIT 10;
```

### 🚨 의도적 오류: 문자열 따옴표 누락

```sql
-- 따옴표 없이 문자열 사용
SELECT * FROM orders
WHERE status = completed;
```

에러 메시지:
```
ERROR: column "completed" does not exist
LINE 2: WHERE status = completed;
                       ^
```

- `completed`를 **컬럼명**으로 인식함
- 문자열은 반드시 **작은따옴표** 필요!

### 올바른 방법

```sql
SELECT * FROM orders
WHERE status = 'completed'
LIMIT 10;
```

### 숫자 조건

```sql
-- 30,000원 이상 주문
SELECT * FROM orders
WHERE total_amount >= 30000
LIMIT 10;
```

```sql
-- 10,000원 ~ 20,000원 사이 주문
SELECT * FROM orders
WHERE total_amount >= 10000 AND total_amount <= 20000
LIMIT 10;
```

```sql
-- BETWEEN 사용 (같은 결과)
SELECT * FROM orders
WHERE total_amount BETWEEN 10000 AND 20000
LIMIT 10;
```

### 여러 조건 조합

```sql
-- AND: 모든 조건 만족
SELECT * FROM orders
WHERE status = 'completed'
  AND total_amount > 30000
LIMIT 10;
```

```sql
-- OR: 하나라도 만족
SELECT * FROM orders
WHERE status = 'completed'
   OR status = 'pending'
LIMIT 10;
```

```sql
-- IN: 여러 값 중 하나 (OR 대신)
SELECT * FROM orders
WHERE status IN ('completed', 'pending')
LIMIT 10;
```

### 날짜 조건

```sql
-- 2025년 12월 주문
SELECT * FROM orders
WHERE created_at >= '2025-12-01'
  AND created_at < '2026-01-01'
LIMIT 10;
```

> **팁**: 날짜 범위는 `>=` 시작일, `<` 다음달 1일 패턴 사용

---
## 4. 연습 문제

### 문제 1
`users` 테이블에서 지역이 '강남'인 고객만 조회하세요.

<details>
<summary>힌트</summary>
WHERE region = '___'
</details>

<details>
<summary>정답</summary>

```sql
SELECT * FROM users
WHERE region = '강남';
```
</details>

### 문제 2
`orders` 테이블에서 총 금액이 40,000원 이상인 주문을 조회하세요.

<details>
<summary>정답</summary>

```sql
SELECT * FROM orders
WHERE total_amount >= 40000
LIMIT 20;
```
</details>

### 문제 3
`orders` 테이블에서 취소된(cancelled) 주문의 개수를 세어보세요.

<details>
<summary>힌트</summary>
COUNT(*) 함수 사용
</details>

<details>
<summary>정답</summary>

```sql
SELECT COUNT(*) FROM orders
WHERE status = 'cancelled';
```
</details>

### 문제 4 (보너스)
2025년 11월에 완료된 주문 중 30,000원 이상인 것만 조회하세요.

<details>
<summary>정답</summary>

```sql
SELECT * FROM orders
WHERE status = 'completed'
  AND total_amount >= 30000
  AND created_at >= '2025-11-01'
  AND created_at < '2025-12-01'
LIMIT 20;
```
</details>

---
## 5. 핵심 정리

| 구문 | 설명 | 예시 |
|------|------|------|
| SELECT * | 모든 컬럼 | `SELECT * FROM users` |
| SELECT 컬럼명 | 특정 컬럼만 | `SELECT name, region FROM users` |
| LIMIT n | 결과 개수 제한 | `LIMIT 10` |
| WHERE = | 같은 값 | `WHERE status = 'completed'` |
| WHERE >= | 이상 | `WHERE total_amount >= 30000` |
| AND / OR | 조건 조합 | `WHERE a = 1 AND b = 2` |
| IN | 여러 값 중 하나 | `WHERE status IN ('a', 'b')` |

### 기억할 것
1. **LIMIT 습관화** - 운영 DB 보호
2. **문자열은 작은따옴표** - `'completed'`
3. **날짜 범위는 >= 와 <** - 경계값 실수 방지

---

## 다음 교시 예고
ORDER BY로 정렬, 집계 함수(COUNT, SUM, AVG)

---


# 3교시: ORDER BY와 집계 함수

## 학습 목표
- ORDER BY로 결과 정렬
- LIMIT + OFFSET으로 페이징
- COUNT, SUM, AVG, MIN, MAX 집계 함수 활용

---

## 1. ORDER BY - 결과 정렬

### 기본 정렬 (오름차순)

```sql
-- 주문금액 낮은 순
SELECT order_id, total_amount, status
FROM orders
ORDER BY total_amount
LIMIT 10;
```

결과:
```
order_id | total_amount | status
---------|--------------|----------
156      | 5230         | completed
289      | 5450         | pending
78       | 5890         | completed
...
```

> 기본값은 **ASC (오름차순)** - 작은 값 → 큰 값

### 내림차순 정렬

```sql
-- 주문금액 높은 순
SELECT order_id, total_amount, status
FROM orders
ORDER BY total_amount DESC
LIMIT 10;
```

결과:
```
order_id | total_amount | status
---------|--------------|----------
423      | 49800        | completed
187      | 49500        | completed
312      | 48900        | pending
...
```

### 최신순 / 오래된순

```sql
-- 최신 주문 먼저
SELECT order_id, created_at, total_amount
FROM orders
ORDER BY created_at DESC
LIMIT 10;
```

```sql
-- 오래된 주문 먼저
SELECT order_id, created_at, total_amount
FROM orders
ORDER BY created_at ASC
LIMIT 10;
```

### 여러 컬럼으로 정렬

```sql
-- 상태별로 정렬하고, 같은 상태 내에서는 금액 높은 순
SELECT order_id, status, total_amount
FROM orders
ORDER BY status, total_amount DESC
LIMIT 20;
```

결과:
```
order_id | status    | total_amount
---------|-----------|-------------
45       | cancelled | 47800
123      | cancelled | 42300
...
256      | completed | 49800
89       | completed | 48500
...
```

---
## 2. LIMIT과 OFFSET - 페이징

### 기본 LIMIT

```sql
-- 상위 5개만
SELECT * FROM orders
ORDER BY total_amount DESC
LIMIT 5;
```

### OFFSET으로 건너뛰기

```sql
-- 1페이지 (1~10번)
SELECT order_id, total_amount
FROM orders
ORDER BY order_id
LIMIT 10 OFFSET 0;

-- 2페이지 (11~20번)
SELECT order_id, total_amount
FROM orders
ORDER BY order_id
LIMIT 10 OFFSET 10;

-- 3페이지 (21~30번)
SELECT order_id, total_amount
FROM orders
ORDER BY order_id
LIMIT 10 OFFSET 20;
```

> **공식**: `OFFSET = (페이지번호 - 1) * LIMIT`

### 연습

**문제**: 가장 최근 주문 5건을 조회하세요.

<details>
<summary>정답</summary>

```sql
SELECT * FROM orders
ORDER BY created_at DESC
LIMIT 5;
```
</details>

**문제**: 주문금액이 가장 낮은 3건을 조회하세요.

<details>
<summary>정답</summary>

```sql
SELECT * FROM orders
ORDER BY total_amount ASC
LIMIT 3;
```
</details>

---
## 3. 집계 함수

### COUNT - 개수 세기

```sql
-- 전체 주문 수
SELECT COUNT(*) FROM orders;
```

결과: `500`

```sql
-- 완료된 주문 수
SELECT COUNT(*) FROM orders
WHERE status = 'completed';
```

결과: 약 `330` (데이터에 따라 다름)

### SUM - 합계

```sql
-- 전체 매출 합계
SELECT SUM(total_amount) FROM orders
WHERE status = 'completed';
```

결과: 약 `8,500,000` 원

### AVG - 평균

```sql
-- 평균 주문금액
SELECT AVG(total_amount) FROM orders
WHERE status = 'completed';
```

결과: 약 `25,700` 원

### MIN, MAX - 최솟값, 최댓값

```sql
-- 최소, 최대 주문금액
SELECT
    MIN(total_amount) as 최소금액,
    MAX(total_amount) as 최대금액
FROM orders;
```

결과:
```
최소금액 | 최대금액
---------|----------
5230     | 49800
```

### 여러 집계 한 번에

```sql
-- 완료된 주문 통계
SELECT
    COUNT(*) as 주문수,
    SUM(total_amount) as 총매출,
    AVG(total_amount) as 평균금액,
    MIN(total_amount) as 최소금액,
    MAX(total_amount) as 최대금액
FROM orders
WHERE status = 'completed';
```

결과:
```
주문수 | 총매출   | 평균금액 | 최소금액 | 최대금액
-------|----------|----------|----------|----------
332    | 8543000  | 25731    | 5230     | 49800
```

### ROUND - 반올림

```sql
-- 평균을 정수로 표시
SELECT ROUND(AVG(total_amount)) as 평균금액
FROM orders
WHERE status = 'completed';
```

```sql
-- 소수점 2자리까지
SELECT ROUND(AVG(total_amount), 2) as 평균금액
FROM orders;
```

---
## 4. 연습 문제

### 문제 1
취소된(cancelled) 주문의 총 금액은 얼마인가요?

<details>
<summary>정답</summary>

```sql
SELECT SUM(total_amount) as 취소총액
FROM orders
WHERE status = 'cancelled';
```
</details>

### 문제 2
전체 주문 중 평균 주문금액보다 비싼 주문은 몇 건인가요?

<details>
<summary>힌트</summary>
먼저 평균을 구하고, WHERE에서 그 값보다 큰 것을 COUNT
</details>

<details>
<summary>정답</summary>

```sql
-- 평균 확인
SELECT AVG(total_amount) FROM orders;  -- 약 25,000

-- 평균보다 비싼 주문 개수
SELECT COUNT(*) FROM orders
WHERE total_amount > (SELECT AVG(total_amount) FROM orders);
```
</details>

### 문제 3
2025년 12월 주문만의 통계를 구하세요. (주문수, 총매출, 평균금액)

<details>
<summary>정답</summary>

```sql
SELECT
    COUNT(*) as 주문수,
    SUM(total_amount) as 총매출,
    ROUND(AVG(total_amount)) as 평균금액
FROM orders
WHERE created_at >= '2025-12-01'
  AND created_at < '2026-01-01';
```
</details>

---
## 5. 핵심 정리

| 구문 | 설명 | 예시 |
|------|------|------|
| ORDER BY 컬럼 | 오름차순 정렬 | `ORDER BY total_amount` |
| ORDER BY 컬럼 DESC | 내림차순 정렬 | `ORDER BY created_at DESC` |
| LIMIT n | 상위 n개 | `LIMIT 10` |
| OFFSET n | n개 건너뛰기 | `LIMIT 10 OFFSET 20` |
| COUNT(*) | 행 개수 | `SELECT COUNT(*) FROM orders` |
| SUM(컬럼) | 합계 | `SELECT SUM(total_amount)` |
| AVG(컬럼) | 평균 | `SELECT AVG(total_amount)` |
| MIN / MAX | 최소 / 최대 | `SELECT MIN(price), MAX(price)` |

### 기억할 것
1. **ORDER BY는 SELECT 마지막에** (WHERE 다음, LIMIT 전)
2. **페이징 공식**: OFFSET = (페이지 - 1) * LIMIT
3. **집계 함수는 단일 값 반환** - 그룹화 없이 사용 시 전체에 대한 값

---

## 다음 교시 예고
GROUP BY로 그룹별 집계, HAVING으로 그룹 조건

---


# 4교시: GROUP BY와 HAVING

## 학습 목표
- GROUP BY로 데이터 그룹화
- 그룹별 집계 (카테고리별 매출, 일별 주문수 등)
- HAVING으로 그룹 조건 지정

---

## 1. GROUP BY - 그룹화

### 상태별 주문 수

```sql
SELECT status, COUNT(*) as 주문수
FROM orders
GROUP BY status;
```

결과:
```
status    | 주문수
----------|-------
completed | 332
pending   | 85
cancelled | 83
```

> GROUP BY는 같은 값을 가진 행들을 **하나의 그룹**으로 묶음

### 🚨 의도적 오류: GROUP BY 규칙 위반

```sql
-- 잘못된 쿼리: GROUP BY 없이 집계와 일반 컬럼 혼합
SELECT status, COUNT(*), total_amount
FROM orders;
```

에러 메시지:
```
ERROR: column "orders.total_amount" must appear in the GROUP BY clause
or be used in an aggregate function
```

### 왜 에러인가?
- `COUNT(*)`는 **모든 행**을 하나의 값으로 집계
- `total_amount`는 **각 행마다** 다른 값
- → 어떤 `total_amount`를 보여줘야 하는지 모호함!

### GROUP BY 규칙
SELECT에 있는 컬럼은 둘 중 하나여야 함:
1. **GROUP BY에 포함**되거나
2. **집계 함수 안에** 있거나

```sql
-- 올바른 방법 1: total_amount도 집계
SELECT status, COUNT(*), SUM(total_amount)
FROM orders
GROUP BY status;

-- 올바른 방법 2: GROUP BY에 포함
SELECT status, total_amount, COUNT(*)
FROM orders
GROUP BY status, total_amount;  -- 이건 의미 없지만 문법상 OK
```

---
## 2. 실용적인 GROUP BY 예제

### 상태별 매출 집계

```sql
SELECT
    status,
    COUNT(*) as 주문수,
    SUM(total_amount) as 총매출,
    ROUND(AVG(total_amount)) as 평균금액
FROM orders
GROUP BY status;
```

결과:
```
status    | 주문수 | 총매출   | 평균금액
----------|--------|----------|----------
completed | 332    | 8543000  | 25731
pending   | 85     | 2187500  | 25735
cancelled | 83     | 2134700  | 25719
```

### 일별 주문 수

```sql
SELECT
    DATE(created_at) as 주문일,
    COUNT(*) as 주문수
FROM orders
GROUP BY DATE(created_at)
ORDER BY 주문일;
```

결과:
```
주문일     | 주문수
-----------|-------
2025-11-01 | 8
2025-11-02 | 12
2025-11-03 | 7
...
```

> `DATE()` 함수로 timestamp에서 날짜만 추출

### 지역별 고객 수

```sql
SELECT region, COUNT(*) as 고객수
FROM users
GROUP BY region
ORDER BY 고객수 DESC;
```

결과:
```
region | 고객수
-------|-------
강남   | 10
서초   | 10
송파   | 10
...
```

---
## 3. HAVING - 그룹 조건

### WHERE vs HAVING

| 구분 | 시점 | 대상 |
|------|------|------|
| WHERE | 그룹화 **전** | 개별 행 |
| HAVING | 그룹화 **후** | 그룹 |

### 주문이 10건 이상인 날만 보기

```sql
SELECT
    DATE(created_at) as 주문일,
    COUNT(*) as 주문수
FROM orders
GROUP BY DATE(created_at)
HAVING COUNT(*) >= 10
ORDER BY 주문일;
```

결과:
```
주문일     | 주문수
-----------|-------
2025-11-02 | 12
2025-11-15 | 11
2025-12-03 | 14
...
```

### WHERE와 HAVING 함께 사용

```sql
-- 완료된 주문만 대상으로, 일별 매출이 500,000원 이상인 날
SELECT
    DATE(created_at) as 주문일,
    COUNT(*) as 주문수,
    SUM(total_amount) as 매출
FROM orders
WHERE status = 'completed'      -- 먼저 완료된 것만 필터
GROUP BY DATE(created_at)
HAVING SUM(total_amount) >= 500000  -- 그룹화 후 매출 조건
ORDER BY 매출 DESC;
```

> **실행 순서**: FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY

---
## 4. 연습 문제

### 문제 1
상태(status)별 총 매출을 구하세요.

<details>
<summary>정답</summary>

```sql
SELECT status, SUM(total_amount) as 총매출
FROM orders
GROUP BY status;
```
</details>

### 문제 2
일별 평균 주문금액을 구하세요.

<details>
<summary>정답</summary>

```sql
SELECT
    DATE(created_at) as 주문일,
    ROUND(AVG(total_amount)) as 평균금액
FROM orders
GROUP BY DATE(created_at)
ORDER BY 주문일;
```
</details>

### 문제 3
주문 건수가 8건 이상인 날짜만 조회하세요.

<details>
<summary>정답</summary>

```sql
SELECT
    DATE(created_at) as 주문일,
    COUNT(*) as 주문수
FROM orders
GROUP BY DATE(created_at)
HAVING COUNT(*) >= 8
ORDER BY 주문일;
```
</details>

### 문제 4 (보너스)
2025년 12월에 완료된 주문만 대상으로, 일별 주문수와 매출을 구하세요.

<details>
<summary>정답</summary>

```sql
SELECT
    DATE(created_at) as 주문일,
    COUNT(*) as 주문수,
    SUM(total_amount) as 매출
FROM orders
WHERE status = 'completed'
  AND created_at >= '2025-12-01'
  AND created_at < '2026-01-01'
GROUP BY DATE(created_at)
ORDER BY 주문일;
```
</details>

---
## 5. 핵심 정리

| 구문 | 설명 | 예시 |
|------|------|------|
| GROUP BY 컬럼 | 컬럼 값으로 그룹화 | `GROUP BY status` |
| GROUP BY 함수 | 함수 결과로 그룹화 | `GROUP BY DATE(created_at)` |
| HAVING 조건 | 그룹에 조건 | `HAVING COUNT(*) > 10` |

### GROUP BY 규칙
SELECT에 있는 컬럼은:
1. GROUP BY에 있거나
2. 집계 함수 안에 있어야 함

### 실행 순서
```
FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

### 기억할 것
1. **WHERE는 개별 행**, HAVING은 그룹
2. **집계 함수 조건은 HAVING**에서
3. **날짜 그룹화**는 `DATE()` 함수 활용

---

## 다음 교시 예고
시나리오 실습: 마케팅팀 요청 분석

---


# 5교시: 시나리오 실습 - 마케팅팀 요청

## 실습 목표
- 지금까지 배운 SELECT, WHERE, GROUP BY, HAVING 종합 활용
- 비즈니스 요청을 SQL로 해결하는 경험

---

## 시나리오

```
📩 From: 마케팅팀 김대리 (Slack)

안녕하세요 데이터팀 신입분~
이번 달(2025년 12월) 주문 현황 정리 좀 부탁드려요!

필요한 것:
1. 이번 달 총 주문 건수
2. 이번 달 총 매출 (완료된 주문만)
3. 일별 주문 건수와 매출 추이
4. 상태별(완료/취소/대기) 주문 비율

급하진 않고 오늘 중으로 부탁드립니다~
```

---
## 실습 시간: 25분

아래 요청사항을 직접 SQL로 작성해보세요.

---

## 요청 1: 이번 달 총 주문 건수

**스스로 시도해보세요!**

<details>
<summary>힌트 1 (10분 후)</summary>

- WHERE로 12월 필터링
- COUNT(*)로 개수 세기
- 날짜 조건: `created_at >= '2025-12-01' AND created_at < '2026-01-01'`
</details>

<details>
<summary>정답</summary>

```sql
SELECT COUNT(*) as 총주문건수
FROM orders
WHERE created_at >= '2025-12-01'
  AND created_at < '2026-01-01';
```
</details>

## 요청 2: 이번 달 총 매출 (완료된 주문만)

<details>
<summary>힌트 1</summary>

- 조건 2개: 12월 + 완료(completed)
- SUM()으로 합계
</details>

<details>
<summary>정답</summary>

```sql
SELECT SUM(total_amount) as 총매출
FROM orders
WHERE created_at >= '2025-12-01'
  AND created_at < '2026-01-01'
  AND status = 'completed';
```
</details>

## 요청 3: 일별 주문 건수와 매출 추이

<details>
<summary>힌트 1</summary>

- GROUP BY로 날짜별 그룹화
- DATE() 함수로 날짜 추출
- ORDER BY로 날짜순 정렬
</details>

<details>
<summary>힌트 2 (거의 답)</summary>

```sql
SELECT
    DATE(created_at) as ___,
    COUNT(*) as ___,
    SUM(total_amount) as ___
FROM orders
WHERE ___ >= '2025-12-01' AND ___ < '2026-01-01'
GROUP BY ___
ORDER BY ___;
```
</details>

<details>
<summary>정답</summary>

```sql
SELECT
    DATE(created_at) as 주문일,
    COUNT(*) as 주문수,
    SUM(total_amount) as 매출
FROM orders
WHERE created_at >= '2025-12-01'
  AND created_at < '2026-01-01'
GROUP BY DATE(created_at)
ORDER BY 주문일;
```
</details>

## 요청 4: 상태별 주문 비율

<details>
<summary>힌트 1</summary>

- GROUP BY status
- 비율 = 해당 상태 건수 / 전체 건수 * 100
</details>

<details>
<summary>정답</summary>

```sql
-- 방법 1: 서브쿼리 사용
SELECT
    status,
    COUNT(*) as 건수,
    ROUND(COUNT(*) * 100.0 / (
        SELECT COUNT(*) FROM orders
        WHERE created_at >= '2025-12-01' AND created_at < '2026-01-01'
    ), 1) as 비율
FROM orders
WHERE created_at >= '2025-12-01'
  AND created_at < '2026-01-01'
GROUP BY status
ORDER BY 건수 DESC;
```

또는 더 간단하게:

```sql
-- 방법 2: 건수만 보여주기 (비율은 수동 계산)
SELECT
    status,
    COUNT(*) as 건수
FROM orders
WHERE created_at >= '2025-12-01'
  AND created_at < '2026-01-01'
GROUP BY status
ORDER BY 건수 DESC;
```
</details>

---
## 리뷰: 마케팅팀에 보낼 결과

### 1. 총 주문 건수
```
총주문건수
----------
약 250건
```

### 2. 총 매출 (완료만)
```
총매출
----------
약 5,500,000원
```

### 3. 일별 추이
```
주문일     | 주문수 | 매출
-----------|--------|----------
2025-12-01 | 8      | 195,000
2025-12-02 | 12     | 287,000
...
```

### 4. 상태별 비율
```
status    | 건수 | 비율(%)
----------|------|--------
completed | 165  | 66.0
pending   | 50   | 20.0
cancelled | 35   | 14.0
```

---
## 오늘 배운 것 복습

| 개념 | 설명 | 예시 |
|------|------|------|
| SELECT | 조회할 컬럼 | `SELECT name, region` |
| WHERE | 행 필터링 | `WHERE status = 'completed'` |
| ORDER BY | 정렬 | `ORDER BY created_at DESC` |
| LIMIT | 결과 제한 | `LIMIT 10` |
| COUNT, SUM, AVG | 집계 | `COUNT(*), SUM(total_amount)` |
| GROUP BY | 그룹화 | `GROUP BY status` |
| HAVING | 그룹 조건 | `HAVING COUNT(*) > 10` |

---

## 내일 예고: JOIN

**오늘의 한계**: "카테고리별 매출을 보고 싶어요"

```sql
-- 이게 안 됨!
SELECT category, SUM(total_amount)
FROM orders
GROUP BY category;
-- ERROR: column "category" does not exist
```

- `orders` 테이블에는 `category`가 없음
- `category`는 `restaurants` 테이블에 있음
- → 두 테이블을 **연결(JOIN)** 해야 함!

```
orders 테이블          restaurants 테이블
┌─────────────────┐    ┌─────────────────────┐
│ order_id        │    │ restaurant_id       │
│ restaurant_id ──┼────┼─→ restaurant_id     │
│ total_amount    │    │ name                │
│ ...             │    │ category ← 여기!   │
└─────────────────┘    └─────────────────────┘
```

**내일 배울 것**: 테이블 연결해서 카테고리별 분석!